# Yeoubi: prepare the byte mixture on Colab

Streams the source datasets, filters and deduplicates them, and writes gzip parts for
`colab_train.ipynb`. Everything downloads inside Google's network, so a slow local
connection only has to carry this page.

- Runtime: **CPU** (Runtime > Change runtime type). No GPU quota is used.
- Edit the settings cell, then Runtime > Run all. Keep the tab open until it prints `done`.
- Output: `parts/part_*.jsonl.gz`, `val.jsonl.gz`, `meta.json`, `dataset_report.json`.

In [ ]:
# SPDX-License-Identifier: Apache-2.0
REPO = "https://github.com/Fury7425/Yeoubi.git"
SCALE = 4.0   # 1.0 = 10B unique bytes (about 4 GB of parts); 4.0 = 40B (about 15 GB)
DEST = "hf"   # "hf" (uses no Drive space) or "drive"
HF_REPO = ""   # private dataset repo to create, such as "your-name/yeoubi-data"
DRIVE_DATA = "/content/drive/MyDrive/yeoubi_data"   # only when DEST == "drive"

In [ ]:
import os
import shutil

assert DEST == "drive" or HF_REPO, "set HF_REPO when DEST is 'hf'"

if not os.path.isdir("/content/Yeoubi"):
    !git clone --depth 1 {REPO} /content/Yeoubi
%cd /content/Yeoubi
!pip -q install -r requirements.txt

# Optional HF_TOKEN from Colab secrets (key icon in the left sidebar). It raises the
# streaming rate limit, and DEST == "hf" needs one with write access.
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    print("no HF_TOKEN secret: streaming anonymously")

# Mount before the build: the sign-in prompt would otherwise block an unattended run
# for hours with the finished data sitting on a disk that is about to be recycled.
if DEST == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    free_gb = shutil.disk_usage("/content/drive/MyDrive").free / 1e9
    print(f"Drive free: {free_gb:.1f} GB, this build needs about {4 * SCALE:.0f} GB")

In [ ]:
%%writefile /content/fast_prep.py
# SPDX-License-Identifier: Apache-2.0
"""Single-pass, multi-process build of the byte mixture for Colab.

prepare_hf_jsonl.py spends most of its time on work this layout does not need: it
stores every byte as a JSON number (about 3.8 characters per byte), compresses that
three times (shard, shuffle bucket, part) and runs on one core. Here each document is
written once, as a {"text": ...} record that train.py's loader encodes itself, and
the sources are read by parallel processes.

Kept: PII redaction and the quality filter from yeoubi.data_pipeline, exact
deduplication across all sources, and a document-level validation split.
Dropped: SimHash near-duplicate detection, chunk overlap and randomized chunk
lengths. The loader cuts a document into consecutive max_seq_len pieces with no
overlap and appends EOS at the document end.

A run always starts from zero; there is no resume.
"""

from __future__ import annotations

import argparse
import gzip
import hashlib
import itertools
import json
import math
import multiprocessing as mp
import queue as queue_module
import random
import re
import shutil
import sys
import time
import traceback
from collections import Counter
from pathlib import Path

sys.path.insert(0, "/content/Yeoubi")
from yeoubi import data_pipeline  # noqa: E402
from yeoubi.data_pipeline import assess_quality, handle_pii  # noqa: E402

# slug, dataset, config, unique bytes at --scale 1, reader processes.
# Mirrors configs/colab_mix.json. Verify every dataset card and license before training.
# The largest source gets two readers so the two vCPUs stay busy until the end.
SOURCES = [
    ("fineweb_edu", "HuggingFaceFW/fineweb-edu", "sample-100BT", 5_000_000_000, 2),
    ("cosmopedia", "HuggingFaceTB/cosmopedia", "web_samples_v1", 3_000_000_000, 1),
    ("wikipedia_en", "wikimedia/wikipedia", "20231101.en", 2_000_000_000, 1),
]
# A resumed training session replays about one part, so parts stay small.
PART_BYTES = 64 * 1024**2
# Every part is an open gzip stream for the whole run; stay under the usual 1024
# descriptor limit. Past this many, parts grow instead.
MAX_PARTS = 900
# Must match --max_seq_len and --batch_size * --grad_accum_steps in colab_train.ipynb:
# meta.json's step counts are derived from them.
MAX_SEQ_LEN = 4096
CHUNKS_PER_STEP = 16
VAL_EVERY = 200   # one document in 200 is held out of training
VAL_DOCS = 3000   # how many of those are stored; train.py reads only a few batches
BATCH_BYTES = 1024**2   # reader-to-writer hand-off size; amortizes the pickling
# Data is compressed exactly once, by the single writer process, so a middling level
# keeps the writer from becoming the bottleneck for little extra size.
GZIP_LEVEL = 4

# The first measured build ran at 1.5 MB/s, nearly all of it inside the four PII regexes
# and the long-run regex: Python's re costs a few hundred ns per character for patterns
# that open with a lookbehind or an alternation, and each one scanned every document.
# Each hint below is a NECESSARY condition for its regex to match and is far cheaper to
# test, so the full regex only runs on documents that could contain a match. The output
# is unchanged; --check asserts that against the original functions.
_IP_HINT = re.compile(r"\d\.\d{1,3}\.\d{1,3}\.\d")   # the inner 3 dots of any dotted quad
# Every phone alternative starts with + ( or a digit, then has at least 6 characters
# from this class before its final digit.
_PHONE_HINT = re.compile(r"[+(\d][\d ()-]{6,}\d")
# The value half of the secret pattern, verbatim; the keyword half is the slow part.
_SECRET_HINT = re.compile(r"(?i)[:=]\s*['\"]?[A-Za-z0-9_./+\-=]{12,}")
# A run of 128 equal characters covers 8 consecutive multiples of 16, so it shows up as
# 8 equal neighbours in every-16th-character sampling, which is 16x less text to scan.
_RUN_HINT = re.compile(r"(.)\1{7}", re.DOTALL)
_LONG_RUN = data_pipeline._LONG_RUN_RE


def redact(text):
    """Same result as handle_pii(text, "redact")[0], skipping scans that cannot match."""
    if "@" in text:
        text = data_pipeline._EMAIL_RE.sub("<EMAIL>", text)
    if _IP_HINT.search(text):
        text = data_pipeline._IPV4_RE.sub("<IP_ADDRESS>", text)
    if _PHONE_HINT.search(text):
        text = data_pipeline._PHONE_RE.sub("<PHONE>", text)
    if _SECRET_HINT.search(text):
        text = data_pipeline._SECRET_RE.sub("<SECRET>", text)
    return text


class _GuardedLongRun:
    """Stands in for data_pipeline._LONG_RUN_RE inside assess_quality."""

    def finditer(self, text):
        if _RUN_HINT.search(text[::16]) is None:
            return iter(())
        return _LONG_RUN.finditer(text)


# ponytail: patches a private name of the cloned package. If data_pipeline changes how
# it finds long runs this silently stops helping; move the guard into assess_quality
# itself once the repo copy is the one being edited.
data_pipeline._LONG_RUN_RE = _GuardedLongRun()


def read_source(slug, path, name, shard, shards, out_queue, stop, check):
    """Stream one shard of a source, filter it, and hand encoded records to the writer."""
    stats = Counter()
    try:
        from datasets import load_dataset   # only reader processes pay for this import

        # batch_size bounds how much of a parquet row group becomes Python objects at once.
        stream = load_dataset(path, name=name, split="train", streaming=True, batch_size=1000)
        if shards > 1:
            if hasattr(stream, "shard"):
                stream = stream.shard(num_shards=shards, index=shard)   # splits by file
            else:
                # Older datasets releases: still correct, but every reader downloads it all.
                stream = itertools.islice(stream, shard, None, shards)
        batch, size = [], 0
        for row in stream:
            if stop.is_set():
                break
            text = row.get("text")
            if not isinstance(text, str):
                continue
            text = text.strip()
            if not text:
                continue
            if check:
                # The rehearsal run proves the shortcuts on real documents before the
                # full build trusts them. A failure here stops the build.
                assert redact(text) == handle_pii(text, "redact")[0], "redact() differs from handle_pii"
                assert _RUN_HINT.search(text[::16]) or not _LONG_RUN.search(text), "long-run guard missed a run"
            text = redact(text)
            quality = assess_quality(text)
            if not quality.accepted:
                stats[f"rejected_{quality.reason}"] += 1
                continue
            try:
                data = text.encode("utf-8")
            except UnicodeEncodeError:
                # A lone surrogate can be neither written here nor encoded by the loader.
                stats["rejected_bad_unicode"] += 1
                continue
            record = json.dumps({"text": text, "source": slug}, ensure_ascii=False)
            line = record.encode("utf-8") + b"\n"
            batch.append((hashlib.blake2b(data, digest_size=8).digest(), line, len(data)))
            size += len(line)
            if size >= BATCH_BYTES:
                out_queue.put((slug, batch, None))
                batch, size = [], 0
        if batch:
            out_queue.put((slug, batch, None))
    except Exception:
        traceback.print_exc()
        stats["reader_failed"] += 1
    finally:
        out_queue.put((slug, None, dict(stats)))   # end marker; the writer counts these


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--scale", type=float, default=1.0)
    parser.add_argument("--out", type=Path, required=True)
    parser.add_argument(
        "--check",
        action="store_true",
        help="Assert the regex shortcuts against the original functions (slow; rehearsal only)",
    )
    args = parser.parse_args()

    targets = {slug: int(target * args.scale) for slug, _, _, target, _ in SOURCES}
    parts = max(1, min(MAX_PARTS, math.ceil(sum(targets.values()) / PART_BYTES)))
    shutil.rmtree(args.out, ignore_errors=True)   # never mix parts from two runs
    parts_dir = args.out / "parts"
    parts_dir.mkdir(parents=True)
    handles = [
        gzip.open(parts_dir / f"part_{index:05d}.jsonl.gz", "wb", compresslevel=GZIP_LEVEL)
        for index in range(parts)
    ]

    out_queue = mp.Queue(maxsize=16)   # readers block while the writer is behind
    stops = {slug: mp.Event() for slug in targets}
    readers = [
        mp.Process(
            target=read_source,
            args=(slug, path, name, shard, shards, out_queue, stops[slug], args.check),
            daemon=True,
        )
        for slug, path, name, _, shards in SOURCES
        for shard in range(shards)
    ]
    for reader in readers:
        reader.start()

    rng = random.Random(42)
    seen = set()   # 8-byte digests: about 1 GB of RAM per 10M documents
    kept, docs, stats = Counter(), Counter(), Counter()
    val_lines = []
    chunks = 0
    running = len(readers)
    started = last_report = time.time()
    while running:
        try:
            slug, batch, reader_stats = out_queue.get(timeout=60)
        except queue_module.Empty:
            if not any(reader.is_alive() for reader in readers):
                # A reader died without its end marker, for example killed for memory.
                stats["reader_failed"] += 1
                break
            continue
        if batch is None:
            running -= 1
            stats.update(reader_stats)
            continue
        for digest, line, size in batch:
            if digest in seen:
                stats["rejected_duplicate"] += 1
                continue
            seen.add(digest)
            if int.from_bytes(digest, "big") % VAL_EVERY == 0:
                # Held out whether or not it is stored, so validation never overlaps training.
                if len(val_lines) < VAL_DOCS:
                    val_lines.append(line)
                continue
            if kept[slug] >= targets[slug]:
                continue   # stragglers from a reader that has not seen the stop yet
            # A random part per document mixes the sources without a shuffle pass.
            handles[rng.randrange(parts)].write(line)
            kept[slug] += size
            docs[slug] += 1
            # The loader appends EOS and cuts every MAX_SEQ_LEN positions.
            chunks += size // MAX_SEQ_LEN + 1
        if kept[slug] >= targets[slug]:
            stops[slug].set()
        now = time.time()
        if now - last_report >= 30:
            last_report = now
            total = sum(kept.values())
            shares = "  ".join(f"{name} {100 * kept[name] // targets[name]}%" for name in targets)
            print(f"{total / 1e9:7.2f} GB  {total / 1e6 / (now - started):5.1f} MB/s  {shares}", flush=True)

    for handle in handles:
        handle.close()
    if stats["reader_failed"]:
        # No meta.json, so the upload cell stops instead of publishing a partial mixture.
        sys.exit("a reader failed (see the traceback above); nothing was finalized")

    rng.shuffle(val_lines)   # stored in arrival order, which is one source at a time
    with gzip.open(args.out / "val.jsonl.gz", "wb", compresslevel=GZIP_LEVEL) as handle:
        handle.writelines(val_lines)
    meta = {
        "parts": parts,
        "steps_per_part": max(1, chunks // parts // CHUNKS_PER_STEP),
        "total_steps": chunks // CHUNKS_PER_STEP,
    }
    (args.out / "meta.json").write_text(json.dumps(meta), encoding="utf-8")
    report = {
        "seconds": round(time.time() - started),
        "target_bytes": targets,
        "unique_bytes": dict(kept),
        "documents": dict(docs),
        "validation_documents": len(val_lines),
        "rejections": dict(stats),
        "meta": meta,
    }
    (args.out / "dataset_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
    for name, target in targets.items():
        if kept[name] < target:
            print(f"note: {name} ran out at {kept[name] / 1e9:.2f} of {target / 1e9:.2f} GB")
    print(json.dumps(report, indent=2))


if __name__ == "__main__":
    main()

In [ ]:
# The first command is a 50 MB rehearsal of the whole script, including the final
# meta.json step, so a mistake shows up in a minute or two and not after the full build.
# --check makes it also compare the fast PII and quality shortcuts with the original
# functions on every document; any difference stops the build before it starts.
# Re-running this cell starts the build from zero.
!python -u /content/fast_prep.py --scale 0.005 --check --out /content/smoke && python -u /content/fast_prep.py --scale {SCALE} --out /content/data/colab

In [ ]:
out = "/content/data/colab"
with open(f"{out}/meta.json", encoding="utf-8") as handle:   # missing if the build failed
    print("meta:", handle.read())

if DEST == "drive":
    shutil.copytree(out, DRIVE_DATA, dirs_exist_ok=True)
    # Drive uploads lazily; without this the runtime can end before the files arrive.
    drive.flush_and_unmount()
else:
    from huggingface_hub import HfApi
    api = HfApi()
    api.create_repo(HF_REPO, repo_type="dataset", private=True, exist_ok=True)
    api.upload_large_folder(repo_id=HF_REPO, repo_type="dataset", folder_path=out)
print("done")